# 05 — Dataset Assembly

Cross-checks `01`'s reconciled points against what `03`/`04` actually
produced on disk (both SVG and TVG graphs must exist and load cleanly),
builds the final training-ready `dataset_index.parquet`, and audits
node/edge-type coverage across the whole dataset before anything gets
trained on it.

**Deliberately NOT done here:** normalization (fit per-fold, inside `07`'s
training loop — computing it globally here would leak fold information
back in) and fold recomputation (`01`'s spatial folds are reused as-is,
just filtered down to the complete points).

In [ ]:
# ── Clone/update repo, mount Drive ──────────────────────────────────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q torch_geometric pyyaml pandas tqdm

In [ ]:
import yaml
from pathlib import Path

with open(f"{REPO_DIR}/configs/paths.yaml") as f:
    paths_cfg = yaml.safe_load(f)

INTERIM_DIR = Path(paths_cfg["interim_dir"])
PROCESSED_DIR = Path(paths_cfg["processed_dir"])

SVG_DIR = PROCESSED_DIR / "svg_graphs"
TVG_DIR = PROCESSED_DIR / "tvg_graphs"
INDEX_OUT = PROCESSED_DIR / "dataset_index.parquet"

In [ ]:
import dataset_audit
import torch
import pandas as pd

reconciled = pd.read_parquet(INTERIM_DIR / "reconciled_points.parquet")
all_point_ids = reconciled["point_id"].tolist()
fold_cols = [c for c in reconciled.columns if c.startswith("fold_rep")]
print(f"Points from 01: {len(all_point_ids)}  |  Fold columns: {fold_cols}")

In [ ]:
# ── Single pass: check existence + load-validity + accumulate type stats ──
from tqdm.auto import tqdm

status_df, node_stats, edge_stats = dataset_audit.scan_dataset(
    tqdm(all_point_ids, desc="Scanning SVG+TVG pairs"), SVG_DIR, TVG_DIR, torch
)

In [ ]:
# ── Drop report — exactly which points are excluded, and why ────────────
report = dataset_audit.build_drop_report(status_df)

print(f"Total points:     {report['total_points']}")
print(f"Complete pairs:   {report['complete_pairs']}")
print(f"Missing SVG only: {report['missing_svg_only']}")
print(f"Missing TVG only: {report['missing_tvg_only']}")
print(f"Missing both:     {report['missing_both']}")

for label, ids in [("missing_svg_only_ids", report["missing_svg_only_ids"]),
                    ("missing_tvg_only_ids", report["missing_tvg_only_ids"]),
                    ("missing_both_ids", report["missing_both_ids"])]:
    if ids:
        print(f"\n{label} ({len(ids)}): {ids[:15]}{'...' if len(ids) > 15 else ''}")

# Also surface load ERRORS specifically (corrupted files), distinct from
# plain missing files — these need re-running 02/03/04 for that point,
# not just acknowledging a gap.
corrupted = status_df[
    (status_df["svg_error"].notna() & (status_df["svg_error"] != "file not found"))
    | (status_df["tvg_error"].notna() & (status_df["tvg_error"] != "file not found"))
]
if len(corrupted):
    print(f"\n⚠️  {len(corrupted)} points had a LOAD ERROR (not just missing) — "
          f"likely truncated files from an interrupted save. Re-run the relevant "
          f"notebook (02/03/04) for these before trusting the count above:")
    display(corrupted[["point_id", "svg_error", "tvg_error"]])

In [ ]:
# ── Build the final index: only points with BOTH graphs loading cleanly ──
final_df = dataset_audit.filter_complete_points(reconciled, status_df)
print(f"Final dataset size: {len(final_df)} / {len(reconciled)} "
      f"({len(reconciled) - len(final_df)} dropped)")

In [ ]:
# ── QC: class balance overall and per fold, AFTER dropping — may have
#    shifted from what 01 originally reported. ──────────────────────────
print(f"Overall: {(final_df['label']==1).sum()} positive, {(final_df['label']==0).sum()} negative")

balance = dataset_audit.fold_balance_report(final_df, fold_cols)
for col, table in balance.items():
    print(f"\n--- {col} ---")
    display(table)

In [ ]:
# ── QC: node/edge-type coverage across the WHOLE dataset — catches a
#    systemic gap (e.g. a node type that never appears anywhere) that
#    per-point QC in 03/04 wouldn't surface. ─────────────────────────────
print("Node type coverage:")
for nt, stats in sorted(node_stats.items()):
    pct = 100 * stats["n_graphs_present"] / len(final_df) if len(final_df) else 0
    flag = "  ⚠️  never appears with content" if stats["n_graphs_present"] == 0 else ""
    print(f"  {nt:20s} total={stats['total_count']:6d}  "
          f"present_in={stats['n_graphs_present']:4d}/{len(final_df)} ({pct:.1f}%){flag}")

print("\nEdge type coverage:")
for ek, stats in sorted(edge_stats.items(), key=lambda kv: str(kv[0])):
    pct = 100 * stats["n_graphs_present"] / len(final_df) if len(final_df) else 0
    flag = "  ⚠️  never appears with content" if stats["n_graphs_present"] == 0 else ""
    print(f"  {str(ek):45s} total={stats['total_count']:6d}  "
          f"present_in={stats['n_graphs_present']:4d}/{len(final_df)} ({pct:.1f}%){flag}")

In [ ]:
# ── Spot-check a few real graphs directly ────────────────────────────────
import random
sample_ids = random.sample(final_df["point_id"].tolist(), min(3, len(final_df)))
for pid in sample_ids:
    svg = torch.load(SVG_DIR / f"{pid}.pt", weights_only=False)
    tvg = torch.load(TVG_DIR / f"{pid}.pt", weights_only=False)
    print(f"\n{pid}:")
    print(f"  SVG node types: {svg.node_types}")
    print(f"  TVG node types: {tvg.node_types}")

In [ ]:
# ── Save the final index ─────────────────────────────────────────────────
final_df.to_parquet(INDEX_OUT, index=False)
print(f"✅ Saved {len(final_df)} rows to {INDEX_OUT}")
print()
print("Next: 06_models.ipynb — training configuration (batch size, epoch cap,")
print("patience, and normalization fitting) still to be decided.")